# Advanced Machine Learning (21CSE425T) — Unit 2: Machine Learning with Graphs

Standalone Colab notebook for **T4–T6**: GNNs with PyTorch Geometric, Knowledge Graphs (rdflib + SPARQL), and Frequent Subgraph Mining (NetworkX). Run top-to-bottom; the setup cell installs all dependencies.

---
# Unit 2 — Machine Learning with Graphs

Installs PyTorch Geometric, NetworkX and rdflib, then covers **Graph Neural Networks** (T4),
**Knowledge Graphs** (T5) and **Frequent Subgraph Mining** (T6).

### Unit 2 — Setup & installs

In [ ]:
# PyTorch Geometric (pure-python install works on Colab; GCNConv needs no C++ extensions).
!pip -q install torch_geometric 2>/dev/null
!pip -q install networkx rdflib 2>/dev/null

import torch, networkx as nx, rdflib
import torch_geometric
print("PyTorch        :", torch.__version__)
print("PyG            :", torch_geometric.__version__)
print("NetworkX       :", nx.__version__)
print("rdflib         :", rdflib.__version__)

# Device handle reused by the PyTorch tasks in this notebook.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## T4 — Building Graph Neural Networks (GCN) with PyTorch Geometric

We train a 2-layer **Graph Convolutional Network** (Kipf & Welling, 2017) on **Zachary's Karate Club** —
a built-in PyG benchmark that needs **no download**. The task is semi-supervised node classification:
only 4 nodes are labelled (one per community) yet the GCN classifies all 34.

In [ ]:
import torch, torch.nn.functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv

torch.manual_seed(0)

dataset = KarateClub()
data = dataset[0].to(DEVICE)
print(data)
print(f"Nodes={data.num_nodes}  Features={dataset.num_features}  "
      f"Classes={dataset.num_classes}  Labelled train nodes={int(data.train_mask.sum())}")

class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden, n_classes):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, n_classes)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))     # message passing + ReLU
        x = F.dropout(x, p=0.3, training=self.training)
        return self.conv2(x, edge_index)          # logits per class

model = GCN(dataset.num_features, 16, dataset.num_classes).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

for epoch in range(1, 201):
    model.train(); optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])  # only labelled nodes
    loss.backward(); optimizer.step()
    if epoch % 40 == 0:
        model.eval()
        pred = model(data.x, data.edge_index).argmax(1)
        acc = (pred == data.y).float().mean().item()      # accuracy over ALL nodes
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | full-graph acc {acc:.3f}")

In [ ]:
# ---- Visualise the graph coloured by the GCN's predicted community ----
import networkx as nx, matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx

model.eval()
pred = model(data.x, data.edge_index).argmax(1).cpu().numpy()
G = to_networkx(data.cpu(), to_undirected=True)

plt.figure(figsize=(7,6))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos, node_color=pred, cmap="tab10",
                 node_size=400, font_size=8, edgecolors="black")
plt.title("T4 — GCN node classification on Karate Club (colour = predicted community)")
plt.axis("off"); plt.tight_layout(); plt.show()

## T5 — Knowledge Graphs: building & querying with rdflib + SPARQL

We build a small RDF **knowledge graph** of people, companies and skills, serialise it to Turtle, then
run **SPARQL** queries (pattern matching and aggregation) over it.

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF
from rdflib.namespace import FOAF, RDFS, XSD

g = Graph()
EX = Namespace("http://example.org/")
g.bind("ex", EX); g.bind("foaf", FOAF)

# ---- Add triples: (subject, predicate, object) ----
people = {
    "Raktim": {"worksAt": "SRM",    "skill": ["Python", "ML"]},
    "Juhi":   {"worksAt": "SRM",    "skill": ["Python", "Research"]},
    "Alice":  {"worksAt": "Acme",   "skill": ["Java"]},
    "Bob":    {"worksAt": "Acme",   "skill": ["Python", "DevOps"]},
}
for name, info in people.items():
    person = EX[name]
    g.add((person, RDF.type, FOAF.Person))
    g.add((person, FOAF.name, Literal(name)))
    g.add((person, EX.worksAt, EX[info["worksAt"]]))
    for s in info["skill"]:
        g.add((person, EX.hasSkill, Literal(s)))
for company in {"SRM", "Acme"}:
    g.add((EX[company], RDF.type, EX.Organization))

print(f"Knowledge graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
# ---- SPARQL query 1: who works at SRM and knows Python? ----
q1 = """
PREFIX ex:   <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE {
    ?p foaf:name ?name .
    ?p ex:worksAt ex:SRM .
    ?p ex:hasSkill "Python" .
}"""
print("SRM employees who know Python:")
for row in g.query(q1):
    print("  -", row.name)

# ---- SPARQL query 2: count employees per organisation (aggregation) ----
q2 = """
PREFIX ex:   <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?org (COUNT(?p) AS ?n) WHERE {
    ?p ex:worksAt ?org .
    ?p a foaf:Person .
} GROUP BY ?org ORDER BY DESC(?n)"""
print("\nHead-count per organisation:")
for row in g.query(q2):
    print(f"  {str(row.org).split('/')[-1]:6s}: {int(row.n)}")

## T6 — Frequent Subgraph Mining (NetworkX)

Given a **database of graphs** (think molecules with atom-typed nodes), we mine connected subgraph
*patterns* that occur in at least `min_support` graphs. We enumerate connected node-subsets, canonicalise
each with the **Weisfeiler–Lehman graph hash** (the WL test underpins modern GNN expressiveness), and count
support across the database — the classic frequent-subgraph-mining workflow, bridged to the GNN world.

In [ ]:
import networkx as nx
from itertools import combinations
from collections import defaultdict

# ---- Graph database: 4 small labelled graphs ----
def make_graph(edges, labels):
    G = nx.Graph()
    for n, lab in labels.items():
        G.add_node(n, label=lab)
    G.add_edges_from(edges)
    return G

db = [
    make_graph([(0,1),(1,2),(2,3)], {0:"C",1:"C",2:"O",3:"H"}),
    make_graph([(0,1),(1,2),(2,3)], {0:"C",1:"C",2:"O",3:"N"}),
    make_graph([(0,1),(1,2)],       {0:"C",1:"C",2:"O"}),
    make_graph([(0,1),(1,2),(1,3)], {0:"C",1:"C",2:"O",3:"C"}),
]
print(f"Graph database: {len(db)} graphs.")

def connected_subgraphs(G, size):
    """Yield all connected node-induced subgraphs of a given size."""
    for nodes in combinations(G.nodes(), size):
        sub = G.subgraph(nodes)
        if nx.is_connected(sub):
            yield sub.copy()

def canonical(sub):
    """WL hash = canonical form that respects node labels (isomorphism-invariant)."""
    return nx.weisfeiler_lehman_graph_hash(sub, node_attr="label")

def mine_frequent_subgraphs(db, min_support=2, sizes=(2, 3)):
    support = defaultdict(set)      # canonical-hash -> set of graph indices it appears in
    example = {}                    # keep one concrete subgraph per hash for display
    for gi, G in enumerate(db):
        seen_here = set()
        for size in sizes:
            for sub in connected_subgraphs(G, size):
                h = canonical(sub)
                if h not in seen_here:          # count each pattern once per graph
                    support[h].add(gi); seen_here.add(h)
                example.setdefault(h, sub)
    frequent = {h: gi_set for h, gi_set in support.items() if len(gi_set) >= min_support}
    return frequent, example

frequent, example = mine_frequent_subgraphs(db, min_support=2, sizes=(2, 3))
print(f"\nFrequent connected subgraph patterns (support >= 2):\n")
for h, gi_set in sorted(frequent.items(), key=lambda kv: -len(kv[1])):
    sub = example[h]
    labels = [sub.nodes[n]["label"] for n in sub.nodes()]
    edges  = [(sub.nodes[u]["label"], sub.nodes[v]["label"]) for u, v in sub.edges()]
    print(f"  support={len(gi_set)}/{len(db)}  nodes={sorted(labels)}  edges(by label)={edges}")